# Part 11: Modern Transformer Architectures — From the 2017 Paper to LLaMA-era LLMs

This is the **capstone** notebook. So far we have built a faithful version of the
2017 *"Attention Is All You Need"* Transformer: learned/sinusoidal positional
embeddings, **LayerNorm** (Pre-LN), a **GELU** feed-forward network, and standard
**multi-head attention**. That design still works — but every large model trained
since ~2020 (GPT-NeoX, PaLM, **LLaMA / LLaMA-2 / LLaMA-3**, Mistral, Qwen, ...)
swaps out four of those components for cheaper, better-scaling replacements.

This notebook builds the **modern decoder block** piece by piece. For each
component we:

1. recall what the repo (`src/`) currently uses,
2. implement the modern replacement inline as a tiny, runnable module,
3. show a concrete numerical check of *why* it is better.

### The mapping: 2017 original → modern (GPT/LLaMA)

| Component            | 2017 original (this repo's `src/`)              | Modern (GPT-NeoX / LLaMA / Mistral)        | Why it changed                                  |
|---------------------|-------------------------------------------------|--------------------------------------------|-------------------------------------------------|
| Normalization        | `nn.LayerNorm` (mean-center + rescale + bias)   | **RMSNorm** (rescale only, no bias)        | Fewer ops, no mean stats, same stability        |
| Position info        | sinusoidal / learned **absolute** PE            | **RoPE** (rotary, applied inside attention)| Relative positions, length extrapolation        |
| Feed-forward         | `FeedForward` = `W2 · GELU(W1 x)`               | **SwiGLU** = `W2 · (silu(W1 x) ⊙ W3 x)`    | Gating → better quality at equal params         |
| Attention            | Multi-Head Attention (`#KV heads = #Q heads`)   | **GQA / MQA** (`#KV heads < #Q heads`)     | Shrinks the KV cache → cheaper inference         |
| Attention *kernel*   | explicit `softmax(QKᵀ) V` (materializes N×N)    | **FlashAttention** (tiled, online softmax) | Same math, far less memory + faster              |

The first four change the *architecture*; FlashAttention is an *implementation
detail* (same math, better IO). We will assemble RMSNorm + RoPE + GQA + SwiGLU
into a single `ModernTransformerBlock` at the end and run it end-to-end.

> Math level: a bit higher than earlier notebooks (this is the advanced one), but
> every claim is backed by runnable code on tiny tensors.

---

In [ ]:
import sys
sys.path.append('..')

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# The repo's 2017-style building blocks, so we can contrast directly.
from src.transformer import FeedForward, DecoderBlock
from src.attention import MultiHeadAttention

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | device = {device}")

%matplotlib inline

## 1. RMSNorm — normalization without the mean

The repo uses `nn.LayerNorm`. For a vector $x \in \mathbb{R}^d$ LayerNorm does:

$$\text{LN}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta,
\qquad \mu = \tfrac{1}{d}\sum_i x_i,\;\; \sigma^2 = \tfrac{1}{d}\sum_i (x_i-\mu)^2 .$$

**RMSNorm** (Zhang & Sennrich, 2019; used by LLaMA) drops the mean-centering
*and* the bias. It only rescales by the root-mean-square:

$$\text{RMS}(x) = \sqrt{\tfrac{1}{d}\sum_i x_i^2 + \epsilon},
\qquad \text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \odot \gamma .$$

Why it works / why it's cheaper:

- **No mean subtraction** → no need to compute $\mu$ or center; empirically the
  re-centering term contributes little to LayerNorm's benefit — the *re-scaling*
  is what stabilizes training.
- **No bias $\beta$** → fewer parameters.
- Fewer reduction passes over the feature dim → a measurable speedup when you do
  it billions of times.

In [ ]:
class RMSNorm(nn.Module):
    '''Root-Mean-Square LayerNorm (LLaMA-style). Rescale only, no mean, no bias.'''
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # gamma, the only param

    def forward(self, x):
        # rms over the last (feature) dimension
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


# --- numerical comparison vs nn.LayerNorm on the same sample tensor ---
d_model = 16
x = torch.randn(2, 4, d_model)

ln = nn.LayerNorm(d_model)         # has weight (gamma) AND bias (beta)
rms = RMSNorm(d_model)             # has weight (gamma) only

with torch.no_grad():
    y_ln = ln(x)
    y_rms = rms(x)

print(f"LayerNorm params : {sum(p.numel() for p in ln.parameters())}  (gamma + beta)")
print(f"RMSNorm  params  : {sum(p.numel() for p in rms.parameters())}  (gamma only)")
print()
print(f"LayerNorm output mean (~0 by construction): {y_ln.mean().item(): .4f}")
print(f"RMSNorm   output mean (NOT centered)      : {y_rms.mean().item(): .4f}")
print(f"LayerNorm output RMS : {y_ln.pow(2).mean().sqrt().item():.4f}")
print(f"RMSNorm   output RMS : {y_rms.pow(2).mean().sqrt().item():.4f}  <- both ~1, scale is preserved")

Note the key difference in the numbers: LayerNorm forces the output mean to ~0
(it subtracts $\mu$); RMSNorm leaves the mean alone and only fixes the scale.
Both keep the RMS at ~1, which is the property that actually matters for
stable gradients. RMSNorm gets there with **half the parameters** and fewer ops.

## 2. Rotary Position Embeddings (RoPE) — relative position via rotation

The repo injects position with **absolute** embeddings *added to the token
embeddings once* (sinusoidal `PositionalEncoding` or learned
`LearnablePositionalEncoding`). Two problems:

- the model only ever sees the absolute index, yet attention really cares about
  *relative* offsets ("3 tokens back");
- a learned table has a fixed `max_seq_len` and cannot extrapolate.

**RoPE** (Su et al., 2021; used by GPT-NeoX, LLaMA, ...) instead rotates the
query and key vectors *inside attention* by an angle proportional to their
position. Pair up the head dimensions $(x_{2i}, x_{2i+1})$ and rotate each pair
by angle $m\theta_i$ at position $m$, with $\theta_i = 10000^{-2i/d}$.

The magic identity: the dot product between a query at position $m$ and a key at
position $n$ depends **only on the relative offset $m-n$**:

$$\langle R_m q,\; R_n k\rangle = \langle R_{m-n}\, q,\; k\rangle .$$

So attention scores become naturally relative, decay smoothly with distance, and
the same rotation rule applies to positions never seen in training.

In [ ]:
def precompute_rope(head_dim, seq_len, base=10000.0, device='cpu'):
    '''Return cos, sin tables of shape (seq_len, head_dim) for RoPE.'''
    assert head_dim % 2 == 0, "RoPE needs an even head_dim"
    # theta_i = base^(-2i/d) for i = 0..d/2-1
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    pos = torch.arange(seq_len, device=device).float()
    angles = torch.outer(pos, inv_freq)              # (seq_len, head_dim/2)
    angles = torch.cat([angles, angles], dim=-1)     # (seq_len, head_dim) duplicate for the rotate-half trick
    return angles.cos(), angles.sin()


def rotate_half(x):
    '''Split last dim in half and rotate: (x1, x2) -> (-x2, x1).'''
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    '''Apply rotary embedding to x of shape (..., seq_len, head_dim).'''
    # cos/sin: (seq_len, head_dim) -> broadcast over batch/head dims
    return x * cos + rotate_half(x) * sin


# --- numerical check: dot product after RoPE depends only on relative offset ---
head_dim, seq_len = 8, 16
cos, sin = precompute_rope(head_dim, seq_len)

q = torch.randn(head_dim)   # a single query vector
k = torch.randn(head_dim)   # a single key vector (same content everywhere)

def rope_score(m, n):
    qm = apply_rope(q, cos[m], sin[m])
    kn = apply_rope(k, cos[n], sin[n])
    return torch.dot(qm, kn).item()

print("Same relative offset (m - n = 2) at different absolute positions:")
for (m, n) in [(2, 0), (5, 3), (10, 8), (15, 13)]:
    print(f"  q@pos{m:2d} . k@pos{n:2d}  (offset {m-n}) -> {rope_score(m, n): .4f}")

print("\nDifferent relative offsets from the same query position (m = 10):")
for n in [10, 9, 7, 3]:
    print(f"  offset {10-n:2d} -> {rope_score(10, n): .4f}")

The first block prints the **same value** for every pair with offset 2,
regardless of the absolute positions — confirming the score is a function of
$m-n$ only. The second block shows the score generally **decays as the offset
grows**, which is exactly the inductive bias we want for language. Contrast this
with the repo's absolute PE, where moving a whole sentence by one token changes
every position vector.

## 3. SwiGLU — a gated feed-forward network

The repo's `FeedForward` is the classic 2-layer MLP with GELU:

$$\text{FFN}(x) = W_2\,\big(\text{GELU}(W_1 x)\big),\qquad W_1: d\to d_{ff},\; W_2: d_{ff}\to d.$$

**SwiGLU** (Shazeer, 2020; used by PaLM and LLaMA) replaces this with a *gated*
unit using **three** weight matrices:

$$\text{SwiGLU}(x) = W_2\,\big(\,\text{silu}(W_1 x)\;\odot\;W_3 x\,\big),
\qquad \text{silu}(z) = z\,\sigma(z).$$

One branch ($W_1$, passed through SiLU/Swish) acts as a learned **gate** on the
other linear branch ($W_3$); the element-wise product lets the network modulate
information multiplicatively. Because it adds a third matrix, the standard
convention is to shrink the hidden dim to $\tfrac{2}{3}\cdot d_{ff}$ so the total
parameter count stays comparable to the 2-matrix GELU MLP.

In [ ]:
class SwiGLU(nn.Module):
    '''Gated FFN: W2( silu(W1 x) * (W3 x) ). LLaMA-style, no biases.'''
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = 4 * dim
        # ~2/3 convention to match the param count of a 2-matrix MLP, rounded to a multiple of 8
        hidden_dim = int(2 * hidden_dim / 3)
        hidden_dim = ((hidden_dim + 7) // 8) * 8
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)  # gate branch (-> SiLU)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)  # value branch
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)  # output projection

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


dim = 64
gelu_ffn = FeedForward(dim)        # repo's W1/W2 GELU MLP, d_ff defaults to 4*dim
swiglu  = SwiGLU(dim)              # 3 matrices, but ~2/3 hidden dim

p_gelu   = sum(p.numel() for p in gelu_ffn.parameters())
p_swiglu = sum(p.numel() for p in swiglu.parameters())

xb = torch.randn(2, 5, dim)
print(f"GELU MLP  hidden = {gelu_ffn.linear1.out_features:4d} | params = {p_gelu:,}")
print(f"SwiGLU    hidden = {swiglu.w1.out_features:4d} | params = {p_swiglu:,}")
print(f"param ratio SwiGLU/GELU = {p_swiglu / p_gelu:.2f}  (~1.0 thanks to the 2/3 rule)")
print(f"SwiGLU output shape: {tuple(swiglu(xb).shape)}  (same as input)")

The two FFNs have nearly identical parameter budgets, but SwiGLU's
multiplicative gating consistently improves quality in practice — which is why
LLaMA-family models adopted it. (The repo's `FeedForward` also has biases; LLaMA
drops them, as we do here.)

## 4. Grouped-Query Attention (GQA) and MQA — shrinking the KV cache

In the repo's `MultiHeadAttention`, the number of **K/V heads equals the number
of Q heads**. During autoregressive generation (the KV-cache trick — every
previously seen key/value is cached so we don't recompute it) that cache has size

$$\text{cache} = 2 \times \text{layers} \times \text{seq\_len} \times \text{n\_kv\_heads} \times \text{head\_dim},$$

and the `n_kv_heads` factor dominates memory and bandwidth at inference time.

- **MQA** (Multi-Query Attention): use **one** shared K/V head for all Q heads.
  Smallest cache, slight quality drop.
- **GQA** (Grouped-Query Attention; LLaMA-2/3, Mistral): a middle ground —
  `n_kv_heads` groups, each shared by several Q heads. Recovers almost all the
  quality of full MHA at a fraction of the cache.

Implementation: project Q to `n_heads` heads but K/V to only `n_kv_heads`, then
**repeat** each K/V head to cover its group of Q heads before attention.

In [ ]:
class GroupedQueryAttention(nn.Module):
    '''Causal self-attention with n_kv_heads <= n_heads. MQA == n_kv_heads=1.

    Uses RoPE on Q/K and torch's scaled_dot_product_attention (FlashAttention) kernel.
    '''
    def __init__(self, dim, n_heads, n_kv_heads):
        super().__init__()
        assert dim % n_heads == 0
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.n_rep = n_heads // n_kv_heads            # how many Q heads share one KV head

        self.wq = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(n_heads * self.head_dim, dim, bias=False)

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads,    self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # RoPE on q and k (cos/sin broadcast over batch & head dims)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        # repeat KV heads to match Q heads (GQA / MQA expansion)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        # FlashAttention kernel: exact, never materializes the T x T matrix
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out)


# --- KV-cache / param comparison: MHA vs GQA vs MQA ---
dim, n_heads = 64, 8
head_dim = dim // n_heads

variants = {
    "MHA  (n_kv=8)": 8,   # full multi-head: KV heads == Q heads
    "GQA  (n_kv=2)": 2,   # grouped: 4 Q heads share each KV head
    "MQA  (n_kv=1)": 1,   # multi-query: single shared KV head
}

cos, sin = precompute_rope(head_dim, 32)
xb = torch.randn(2, 10, dim)
print(f"{'variant':14s} {'KV-heads':>9s} {'KV-cache/tok':>13s} {'rel. cache':>11s} {'#params':>10s}")
for name, n_kv in variants.items():
    attn = GroupedQueryAttention(dim, n_heads, n_kv)
    _ = attn(xb, cos, sin)                                   # forward works
    kv_per_tok = 2 * n_kv * head_dim                          # K and V per token, per layer
    rel = n_kv / n_heads
    nparams = sum(p.numel() for p in attn.parameters())
    print(f"{name:14s} {n_kv:9d} {kv_per_tok:13d} {rel:11.2f} {nparams:10,d}")

print("\nMQA stores 8x fewer K/V values per token than full MHA -> 8x smaller KV cache.")

The `KV-cache/tok` column is what matters at inference: MQA caches **8×
less** key/value data than full MHA here, and GQA-2 caches 4× less. For long
contexts and big batches the KV cache often dwarfs the weights in memory, so this
is a major inference win — at near-identical quality. (Notice the Q/output
projections are unchanged; only the K/V projections shrink.)

## 5. FlashAttention — same math, IO-aware kernel

Everything above changed the *architecture*. FlashAttention (Dao et al., 2022)
changes only the *kernel* used to compute attention — the math is **exactly the
same** $\text{softmax}(QK^\top/\sqrt{d})V$.

The naive implementation (like the repo's `ScaledDotProductAttention`)
materializes the full $N\times N$ score matrix in GPU memory, costing $O(N^2)$
memory and a lot of slow reads/writes to high-bandwidth memory (HBM).
FlashAttention instead:

- **tiles** Q, K, V into blocks that fit in fast on-chip SRAM,
- uses an **online softmax** (running max + running sum) so it never needs the
  whole row of scores at once,
- thus **never materializes the $N\times N$ matrix** — memory drops to $O(N)$,
  and avoiding HBM round-trips gives a large wall-clock speedup.

It is an **exact** optimization (not an approximation like sparse/linear
attention), which is why it became the default everywhere. In PyTorch you get it
for free via `F.scaled_dot_product_attention` — that is exactly what our
`GroupedQueryAttention` above already calls.

In [ ]:
# The practical one-liner: PyTorch picks a fused (FlashAttention-style) kernel.
B, H, T, D = 2, 4, 32, 16
q = torch.randn(B, H, T, D)
k = torch.randn(B, H, T, D)
v = torch.randn(B, H, T, D)

# (a) fused kernel - never builds the full T x T matrix
fast = F.scaled_dot_product_attention(q, k, v, is_causal=True)

# (b) manual reference - materializes the T x T score matrix (what FlashAttention avoids)
scores = (q @ k.transpose(-2, -1)) / math.sqrt(D)
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
scores = scores.masked_fill(mask, float('-inf'))
ref = torch.softmax(scores, dim=-1) @ v

print(f"explicit score matrix shape : {tuple(scores.shape)}  <- FlashAttention never builds this")
print(f"max |fused - reference|     : {(fast - ref).abs().max().item():.2e}  (identical up to fp error)")

## 6. Assembling the Modern Block

Now we combine all four upgrades into one **`ModernTransformerBlock`** and compare
it to the repo's `DecoderBlock`:

| | repo `DecoderBlock` | `ModernTransformerBlock` |
|---|---|---|
| Norm | `nn.LayerNorm` ×2 | **RMSNorm** ×2 |
| Position | absolute PE added at embedding | **RoPE** inside attention |
| Attention | MHA (`#KV = #Q`) | **GQA** (`#KV < #Q`) + Flash kernel |
| FFN | GELU MLP | **SwiGLU** |
| Structure | Pre-norm + residuals | Pre-norm + residuals (unchanged) |

The skeleton — *pre-norm, attention, residual, pre-norm, FFN, residual* — is
deliberately identical. The 2017 macro-architecture survived; only the components
inside it were upgraded.

In [ ]:
class ModernTransformerBlock(nn.Module):
    '''LLaMA-style decoder block: RMSNorm + RoPE + GQA + SwiGLU, pre-norm + residuals.'''
    def __init__(self, dim, n_heads, n_kv_heads, ffn_hidden=None):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = GroupedQueryAttention(dim, n_heads, n_kv_heads)
        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, ffn_hidden)

    def forward(self, x, cos, sin):
        # Pre-norm + residual around attention
        x = x + self.attn(self.attn_norm(x), cos, sin)
        # Pre-norm + residual around the SwiGLU FFN
        x = x + self.ffn(self.ffn_norm(x))
        return x


# --- end-to-end forward pass on a toy input ---
dim, n_heads, n_kv_heads = 64, 8, 2     # GQA: 4 Q heads per KV head
seq_len = 12
head_dim = dim // n_heads

block = ModernTransformerBlock(dim, n_heads, n_kv_heads).to(device)
cos, sin = precompute_rope(head_dim, seq_len, device=device)
cos, sin = cos.to(device), sin.to(device)

x = torch.randn(2, seq_len, dim, device=device)
out = block(x, cos, sin)

print(f"input  shape : {tuple(x.shape)}")
print(f"output shape : {tuple(out.shape)}")
print(f"finite output: {torch.isfinite(out).all().item()}")
print(f"block params : {sum(p.numel() for p in block.parameters()):,}")
assert out.shape == x.shape and torch.isfinite(out).all()
print("\nModernTransformerBlock forward pass OK.")

In [ ]:
# Side-by-side parameter comparison vs the repo's 2017-style DecoderBlock.
repo_block = DecoderBlock(d_model=dim, num_heads=n_heads, max_seq_len=seq_len)
p_repo   = sum(p.numel() for p in repo_block.parameters())
p_modern = sum(p.numel() for p in block.parameters())
print(f"repo DecoderBlock (LayerNorm + abs-PE + MHA + GELU) : {p_repo:,} params")
print(f"ModernTransformerBlock (RMSNorm+RoPE+GQA+SwiGLU)     : {p_modern:,} params")
print("\nSimilar budget, but the modern block trains/serves more efficiently:")
print(" - RMSNorm: fewer norm params & ops")
print(" - GQA: ~4x smaller KV cache at inference (n_kv=2 vs n_heads=8)")
print(" - RoPE: relative positions, extrapolates beyond trained length")
print(" - SwiGLU: better quality per parameter")

## 7. Scaling & what's next

The architecture is only half the story — the other half is *how big* and *on how
much data* you train.

- **Scaling laws** (Kaplan et al., 2020): loss falls as a smooth power law in
  model **parameters**, **data**, and **compute**. Bigger + more data + more
  compute predictably helps.
- **Chinchilla-optimal** (Hoffmann et al., 2022): for a fixed compute budget,
  the earlier GPT-3-era models were *too big and undertrained*. The compute-
  optimal recipe scales params and tokens together — roughly **~20 training
  tokens per parameter**. Modern open models often train far past that
  ("over-train") to get cheaper inference per quality.
- **Mixture-of-Experts (MoE)**: replace the dense FFN with many expert FFNs and a
  router that activates only a few per token — more *total* parameters but
  constant *active* compute (Mixtral, DeepSeek-V3, GPT-OSS).
- **Long context**: RoPE makes this tractable, plus tricks like
  **position interpolation / NTK-aware** & **YaRN** scaling, sliding-window /
  local attention (Mistral), and FlashAttention to keep the memory linear.

### Summary

We took the 2017 block apart and rebuilt it the modern way, swapping four
components and one kernel — then proved each upgrade with tiny runnable code:

| Upgrade | Module we built | Win demonstrated |
|---|---|---|
| LayerNorm → **RMSNorm** | `RMSNorm` | half the params, same RMS≈1 stabilization |
| absolute PE → **RoPE** | `apply_rope` | dot product depends only on relative offset |
| GELU MLP → **SwiGLU** | `SwiGLU` | gating at ~equal param count (2/3 rule) |
| MHA → **GQA / MQA** | `GroupedQueryAttention` | up to 8× smaller KV cache |
| naive attn → **FlashAttention** | `F.scaled_dot_product_attention` | exact, no N×N matrix |

…all wired together in `ModernTransformerBlock`, which ran end-to-end.

### Key Takeaways

- The **macro-architecture from 2017 survived**: pre-norm Transformer block,
  attention + FFN with residuals. Modern LLMs mostly **swap the components**.
- Most modern changes target **efficiency** — fewer params/ops (RMSNorm),
  cheaper inference (GQA, FlashAttention), better positions (RoPE) — while
  SwiGLU buys quality at equal cost.
- **FlashAttention is exact**, not an approximation: same outputs, far less
  memory and time. It's the default kernel in `torch`'s SDPA.
- A `ModernTransformerBlock` is essentially a **LLaMA layer**; stack N of them
  with a tied embedding/LM head (notebook 07) and you have a modern LLM.

### Where to go next

- **[nanoGPT](https://github.com/karpathy/nanoGPT)** — a clean, minimal,
  trainable GPT; a great next step after this repo. (See also
  [llama2.c](https://github.com/karpathy/llama2.c) for a LLaMA-style version with
  the exact RMSNorm + RoPE + SwiGLU stack we built here.)
- **Train on a real dataset** — swap the toy text from notebook 07 for something
  larger (TinyStories, WikiText, or your own corpus) and watch the loss follow
  the scaling laws.
- **Re-read the references in the repo README** — the original *Attention Is All
  You Need* paper, *The Illustrated Transformer*, and the GPT paper — now with
  the modern stack as context.

You've gone from the 2017 paper to a LLaMA-era block. That's the whole journey.